Huấn luyện + Validation + Test mạng 2D-CNN cho bộ STFT cân bằng Nw=256

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import copy
import random
import time
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from tqdm.auto import tqdm

In [ ]:
# Cấu hình
DATA_ROOT = Path(
    '/content/drive/MyDrive/'
    'balanced_stft_49_each_class_Nw256/'
    'stft_dataset'
)

RESULT_ROOT = Path(
    '/content/drive/MyDrive/'
    'balanced_stft_49_each_class_Nw256/'
    'cnn_results'
)

VARIANT_NAME = 'Nw_256_H_30_129x129'
IMAGE_SIZE = 129

CLASS_TO_LABEL = {
    'normal': 0,
    'inner_race': 1,
    'outer_race': 2,
    'ball_fault': 3,
}

CLASS_NAMES = [
    'Normal',
    'Inner race',
    'Outer race',
    'Ball fault',
]

BATCH_SIZE = 2
EPOCHS = 30
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0
EARLY_STOPPING_PATIENCE = 8
RANDOM_SEED = 42
NUM_WORKERS = 0

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PIN_MEMORY = DEVICE.type == 'cuda'

RESULT_ROOT.mkdir(parents=True, exist_ok=True)

print('DATA_ROOT     :', DATA_ROOT)
print('RESULT_ROOT   :', RESULT_ROOT)
print('DEVICE        :', DEVICE)
print('IMAGE_SIZE    :', IMAGE_SIZE)
print('BATCH_SIZE    :', BATCH_SIZE)
print('EPOCHS        :', EPOCHS)
print('LEARNING_RATE :', LEARNING_RATE)
print('PATIENCE      :', EARLY_STOPPING_PATIENCE)

if DEVICE.type == 'cuda':
    print('GPU           :', torch.cuda.get_device_name(0))

In [ ]:
# Cố định random seed

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(RANDOM_SEED)

In [ ]:
# Quét dữ liệu

def collect_records(split_name):
    records = []
    split_root = DATA_ROOT / split_name

    if not split_root.exists():
        raise FileNotFoundError(f'Không tìm thấy: {split_root}')

    for class_key, label in CLASS_TO_LABEL.items():
        class_root = split_root / class_key
        if not class_root.exists():
            raise FileNotFoundError(f'Không tìm thấy: {class_root}')

        files = []
        for npy_path in class_root.rglob('segment_*.npy'):
            if npy_path.parent.name == VARIANT_NAME:
                files.append(npy_path)

        files = sorted(files)

        for npy_path in files:
            records.append({
                'path': npy_path,
                'label': label,
                'class_key': class_key,
            })

    return sorted(records, key=lambda r: (r['label'], str(r['path'])))


def class_counts(records):
    counts = {name: 0 for name in CLASS_NAMES}
    for record in records:
        counts[CLASS_NAMES[record['label']]] += 1
    return counts


train_records = collect_records('train')
val_records = collect_records('validation')
test_records = collect_records('test')

print('Train      :', len(train_records), class_counts(train_records))
print('Validation :', len(val_records), class_counts(val_records))
print('Test       :', len(test_records), class_counts(test_records))

In [ ]:
# Kiểm tra cân bằng lớp
for split_name, records in [
    ('Train', train_records),
    ('Validation', val_records),
    ('Test', test_records),
]:
    counts = class_counts(records)
    if len(set(counts.values())) == 1:
        print(f'{split_name}: cân bằng lớp')
    else:
        print(f'CẢNH BÁO - {split_name}: lệch lớp -> {counts}')

In [ ]:
# Dataset
class STFTDataset(Dataset):
    def __init__(self, records, image_size):
        self.records = records
        self.image_size = image_size

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        x = np.load(record['path'], allow_pickle=False).astype(np.float32)

        expected_shape = (self.image_size, self.image_size)
        if x.shape != expected_shape:
            raise ValueError(
                f'Sai shape {x.shape}; cần {expected_shape}\n{record["path"]}'
            )

        if not np.all(np.isfinite(x)):
            raise ValueError(f'NaN/Inf tại:\n{record["path"]}')

        x_tensor = torch.from_numpy(x).unsqueeze(0)  # [H,W] -> [1,H,W]
        y_tensor = torch.tensor(record['label'], dtype=torch.long)
        return x_tensor, y_tensor

In [ ]:
# DataLoader
train_dataset = STFTDataset(train_records, IMAGE_SIZE)
val_dataset = STFTDataset(val_records, IMAGE_SIZE)
test_dataset = STFTDataset(test_records, IMAGE_SIZE)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
)

print('Train batches      :', len(train_loader))
print('Validation batches :', len(val_loader))
print('Test batches       :', len(test_loader))

In [ ]:
# Mạng 2D-CNN 4 khối
class Bearing2DCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 4
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        return self.classifier(x)

In [ ]:
# Khởi tạo model
model = Bearing2DCNN(num_classes=4).to(DEVICE)

trainable_parameters = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print(model)
print('\nSố tham số có thể học:', f'{trainable_parameters:,}')

In [ ]:
# Loss + Optimizer
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

print('Loss      : CrossEntropyLoss')
print('Optimizer : Adam')
print('LR        :', LEARNING_RATE)

In [ ]:
# Hàm đánh giá Validation/Test

def evaluate_loader(model, loader, description='Validation', show_progress=False):
    model.eval()

    total_loss = 0.0
    total_samples = 0
    y_true = []
    y_pred = []

    iterator = tqdm(loader, desc=description, leave=False) if show_progress else loader

    with torch.no_grad():
        for x, y in iterator:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            logits = model(x)
            loss = criterion(logits, y)
            predictions = logits.argmax(dim=1)

            B = y.size(0)
            total_loss += loss.item() * B
            total_samples += B

            y_true.extend(y.cpu().numpy().tolist())
            y_pred.extend(predictions.cpu().numpy().tolist())

    loss_value = total_loss / total_samples
    accuracy = accuracy_score(y_true, y_pred)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3],
        average='macro',
        zero_division=0,
    )

    precision_class, recall_class, f1_class, support_class = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3],
        average=None,
        zero_division=0,
    )

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3])

    per_class_df = pd.DataFrame({
        'Class': CLASS_NAMES,
        'Precision': precision_class,
        'Recall': recall_class,
        'F1': f1_class,
        'Support': support_class,
    })

    return {
        'loss': float(loss_value),
        'accuracy': float(accuracy),
        'precision_macro': float(precision_macro),
        'recall_macro': float(recall_macro),
        'f1_macro': float(f1_macro),
        'confusion_matrix': cm,
        'per_class': per_class_df,
        'y_true': y_true,
        'y_pred': y_pred,
    }

In [ ]:
# Huấn luyện
best_val_loss = float('inf')
best_epoch = 0
best_state = None
patience_counter = 0
history = []

training_start = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.perf_counter()
    model.train()

    total_train_loss = 0.0
    total_train_correct = 0
    total_train_samples = 0

    progress_bar = tqdm(
        train_loader,
        desc=f'Epoch {epoch:02d}/{EPOCHS} - Train',
        leave=False,
    )

    for x, y in progress_bar:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        predictions = logits.argmax(dim=1)
        B = y.size(0)

        total_train_loss += loss.item() * B
        total_train_correct += (predictions == y).sum().item()
        total_train_samples += B

        current_loss = total_train_loss / total_train_samples
        current_acc = total_train_correct / total_train_samples

        progress_bar.set_postfix(
            loss=f'{current_loss:.4f}',
            acc=f'{current_acc*100:.2f}%'
        )

    train_loss = total_train_loss / total_train_samples
    train_accuracy = total_train_correct / total_train_samples

    val_result = evaluate_loader(model, val_loader)
    epoch_time = time.perf_counter() - epoch_start

    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'train_accuracy': train_accuracy,
        'val_loss': val_result['loss'],
        'val_accuracy': val_result['accuracy'],
        'val_precision_macro': val_result['precision_macro'],
        'val_recall_macro': val_result['recall_macro'],
        'val_f1_macro': val_result['f1_macro'],
        'epoch_time_sec': epoch_time,
    })

    print(
        f'Epoch {epoch:02d}/{EPOCHS} | '
        f'Train Loss={train_loss:.5f} | '
        f'Train Acc={train_accuracy*100:6.2f}% | '
        f'Val Loss={val_result["loss"]:.5f} | '
        f'Val Acc={val_result["accuracy"]*100:6.2f}% | '
        f'Val F1={val_result["f1_macro"]*100:6.2f}% | '
        f'Time={epoch_time:.1f}s'
    )

    # Best model theo Validation Loss nhỏ nhất
    if val_result['loss'] < best_val_loss:
        best_val_loss = val_result['loss']
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
        print('  -> Lưu best model mới.')
    else:
        patience_counter += 1

    if patience_counter >= EARLY_STOPPING_PATIENCE:
        print(f'Early stopping tại epoch {epoch}.')
        break

training_time = time.perf_counter() - training_start
print('\nTổng thời gian train:', f'{training_time/60:.2f} phút')

In [ ]:
# Khôi phục và lưu Best Model
if best_state is None:
    raise RuntimeError('Không có best model.')

model.load_state_dict(best_state)

best_model_path = RESULT_ROOT / 'best_model.pt'
torch.save(model.state_dict(), best_model_path)

print('Best epoch :', best_epoch)
print('Best model :', best_model_path)

In [ ]:
# Lưu history
history_df = pd.DataFrame(history)
history_path = RESULT_ROOT / 'training_history.csv'
history_df.to_csv(history_path, index=False, encoding='utf-8-sig')

display(history_df)
print('Đã lưu:', history_path)

In [ ]:
# Vẽ Accuracy và Loss
figure, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(history_df['epoch'], history_df['train_accuracy'], label='Train_acc')
axes[0].plot(history_df['epoch'], history_df['val_accuracy'], label='Val_acc')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0.88, 1.005)
axes[0].legend()

axes[1].plot(history_df['epoch'], history_df['train_loss'], label='Train_loss')
axes[1].plot(history_df['epoch'], history_df['val_loss'], label='Val_loss')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Loss')
axes[1].legend()

figure.tight_layout()
figure.savefig(RESULT_ROOT / 'training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Validation với Best Model
best_val_result = evaluate_loader(
    model,
    val_loader,
    description='Validation',
    show_progress=True,
)

print('Validation Loss     :', f'{best_val_result["loss"]:.6f}')
print('Validation Accuracy :', f'{best_val_result["accuracy"]*100:.4f}%')
print('Validation Macro F1 :', f'{best_val_result["f1_macro"]*100:.4f}%')
display(best_val_result['per_class'])

In [ ]:
# Validation Confusion Matrix chuẩn hóa
val_cm_raw = best_val_result['confusion_matrix']
val_cm_norm = val_cm_raw / (val_cm_raw.sum(axis=1, keepdims=True) + 1e-12)

display_cm = ConfusionMatrixDisplay(
    confusion_matrix=val_cm_norm,
    display_labels=CLASS_NAMES,
)

figure, axis = plt.subplots(figsize=(6, 5))
display_cm.plot(
    ax=axis,
    values_format='.2f',
    xticks_rotation=30,
    colorbar=True,
)
axis.set_title('Normalized Validation Confusion Matrix')
figure.tight_layout()
figure.savefig(
    RESULT_ROOT / 'validation_confusion_matrix.png',
    dpi=300,
    bbox_inches='tight',
)
plt.show()

In [ ]:
# Test bằng Best Model
# Test chỉ chạy sau khi đã chọn xong Best Model bằng Validation.
test_start = time.perf_counter()

test_result = evaluate_loader(
    model,
    test_loader,
    description='Test',
    show_progress=True,
)

test_time = time.perf_counter() - test_start

print('\nTEST RESULT')
print('Test Loss            :', f'{test_result["loss"]:.6f}')
print('Test Accuracy        :', f'{test_result["accuracy"]*100:.4f}%')
print('Test Macro Precision :', f'{test_result["precision_macro"]*100:.4f}%')
print('Test Macro Recall    :', f'{test_result["recall_macro"]*100:.4f}%')
print('Test Macro F1        :', f'{test_result["f1_macro"]*100:.4f}%')
print('Test time            :', f'{test_time:.2f} s')

display(test_result['per_class'])


## So sánh Train và Test bằng cùng Best Model

Phần này **không dùng Test trong quá trình học**.

Sau khi Validation đã chọn xong `best_model.pt`, cùng một Best Model được chạy lại trên:

\[
Train \rightarrow Best\ Model \rightarrow Metrics
\]

và:

\[
Test \rightarrow Best\ Model \rightarrow Metrics
\]

Sau đó mới vẽ đồ thị so sánh. Vì vậy đồ thị này không gây rò rỉ dữ liệu từ Test vào quá trình cập nhật trọng số.

Các chỉ số so sánh:

- Accuracy
- Macro Precision
- Macro Recall
- Macro F1


In [ ]:

# So sánh Train và Test bằng Best Model

# Đánh giá lại toàn bộ Train bằng model.eval()
# để Train và Test được đo trong cùng điều kiện.
train_eval_result = evaluate_loader(
    model,
    train_loader,
    description='Train evaluation',
    show_progress=True,
)

print('\nTRAIN EVALUATION WITH BEST MODEL')
print('Train Loss            :', f'{train_eval_result["loss"]:.6f}')
print('Train Accuracy        :', f'{train_eval_result["accuracy"]*100:.4f}%')
print('Train Macro Precision :', f'{train_eval_result["precision_macro"]*100:.4f}%')
print('Train Macro Recall    :', f'{train_eval_result["recall_macro"]*100:.4f}%')
print('Train Macro F1        :', f'{train_eval_result["f1_macro"]*100:.4f}%')

print('\nTEST EVALUATION WITH BEST MODEL')
print('Test Loss             :', f'{test_result["loss"]:.6f}')
print('Test Accuracy         :', f'{test_result["accuracy"]*100:.4f}%')
print('Test Macro Precision  :', f'{test_result["precision_macro"]*100:.4f}%')
print('Test Macro Recall     :', f'{test_result["recall_macro"]*100:.4f}%')
print('Test Macro F1         :', f'{test_result["f1_macro"]*100:.4f}%')


In [ ]:

# Đồ thị so sánh Train và Test:
# Accuracy, Macro Precision, Macro Recall, Macro F1

comparison_df = pd.DataFrame({
    'Metric': [
        'Accuracy',
        'Precision',
        'Recall',
        'F1-score',
    ],
    'Train': [
        train_eval_result['accuracy'],
        train_eval_result['precision_macro'],
        train_eval_result['recall_macro'],
        train_eval_result['f1_macro'],
    ],
    'Test': [
        test_result['accuracy'],
        test_result['precision_macro'],
        test_result['recall_macro'],
        test_result['f1_macro'],
    ],
})

display(comparison_df)

comparison_df.to_csv(
    RESULT_ROOT / 'train_test_comparison.csv',
    index=False,
    encoding='utf-8-sig',
)

x = np.arange(len(comparison_df))
width = 0.36

figure, axis = plt.subplots(
    figsize=(8, 5)
)

axis.bar(
    x - width / 2,
    comparison_df['Train'],
    width,
    label='Train',
)

axis.bar(
    x + width / 2,
    comparison_df['Test'],
    width,
    label='Test',
)

axis.set_xticks(x)
axis.set_xticklabels(
    comparison_df['Metric']
)

axis.set_ylabel(
    'Score'
)

axis.set_xlabel(
    'Evaluation metrics'
)

axis.set_ylim(
    0.0,
    1.05
)

axis.set_title(
    'Comparison of Train and Test performance'
)

axis.legend()

# Ghi giá trị lên từng cột
for container in axis.containers:
    axis.bar_label(
        container,
        fmt='%.4f',
        padding=3,
    )

figure.tight_layout()

figure.savefig(
    RESULT_ROOT / 'train_test_metrics_comparison.png',
    dpi=300,
    bbox_inches='tight',
)

plt.show()


In [ ]:

# Đồ thị so sánh Loss giữa Train và Test

loss_df = pd.DataFrame({
    'Dataset': [
        'Train',
        'Test',
    ],
    'Loss': [
        train_eval_result['loss'],
        test_result['loss'],
    ],
})

display(loss_df)

figure, axis = plt.subplots(
    figsize=(5, 4)
)

bars = axis.bar(
    loss_df['Dataset'],
    loss_df['Loss'],
)

axis.set_ylabel(
    'Cross-Entropy Loss'
)

axis.set_xlabel(
    'Dataset'
)

axis.set_title(
    'Comparison of Train and Test loss'
)

axis.bar_label(
    bars,
    fmt='%.6f',
    padding=3,
)

figure.tight_layout()

figure.savefig(
    RESULT_ROOT / 'train_test_loss_comparison.png',
    dpi=300,
    bbox_inches='tight',
)

plt.show()


In [ ]:
# Test Confusion Matrix chuẩn hóa
test_cm_raw = test_result['confusion_matrix']
test_cm_norm = test_cm_raw / (test_cm_raw.sum(axis=1, keepdims=True) + 1e-12)

display_cm = ConfusionMatrixDisplay(
    confusion_matrix=test_cm_norm,
    display_labels=CLASS_NAMES,
)

figure, axis = plt.subplots(figsize=(6, 5))
display_cm.plot(
    ax=axis,
    values_format='.2f',
    xticks_rotation=30,
    colorbar=True,
)
axis.set_title('Normalized Test Confusion Matrix')
figure.tight_layout()
figure.savefig(
    RESULT_ROOT / 'test_confusion_matrix.png',
    dpi=300,
    bbox_inches='tight',
)
plt.show()

np.savetxt(
    RESULT_ROOT / 'test_confusion_matrix_raw.csv',
    test_cm_raw,
    delimiter=',',
    fmt='%d',
)

np.savetxt(
    RESULT_ROOT / 'test_confusion_matrix_normalized.csv',
    test_cm_norm,
    delimiter=',',
    fmt='%.6f',
)

In [ ]:
# Lưu metrics cuối cùng
test_result['per_class'].to_csv(
    RESULT_ROOT / 'test_per_class_metrics.csv',
    index=False,
    encoding='utf-8-sig',
)

summary = {
    'train_eval_loss': float(train_eval_result['loss']),
    'train_eval_accuracy': float(train_eval_result['accuracy']),
    'train_eval_precision_macro': float(train_eval_result['precision_macro']),
    'train_eval_recall_macro': float(train_eval_result['recall_macro']),
    'train_eval_f1_macro': float(train_eval_result['f1_macro']),
    'variant': VARIANT_NAME,
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'best_epoch': best_epoch,
    'training_time_sec': float(training_time),
    'validation_loss': float(best_val_result['loss']),
    'validation_accuracy': float(best_val_result['accuracy']),
    'validation_f1_macro': float(best_val_result['f1_macro']),
    'test_loss': float(test_result['loss']),
    'test_accuracy': float(test_result['accuracy']),
    'test_precision_macro': float(test_result['precision_macro']),
    'test_recall_macro': float(test_result['recall_macro']),
    'test_f1_macro': float(test_result['f1_macro']),
}

with open(RESULT_ROOT / 'final_metrics.json', 'w', encoding='utf-8') as file:
    json.dump(summary, file, ensure_ascii=False, indent=2)

print('Hoàn thành Train + Validation + Test.')
print('Kết quả lưu tại:', RESULT_ROOT)

In [ ]:
# Xem Softmax của một mẫu Test cụ thể
sample_index = 0

if not (0 <= sample_index < len(test_dataset)):
    raise IndexError(
        f'sample_index phải từ 0 đến {len(test_dataset)-1}'
    )

x, y = test_dataset[sample_index]
record = test_records[sample_index]

x = x.unsqueeze(0).to(DEVICE)
model.eval()

with torch.no_grad():
    logits = model(x)
    probabilities = torch.softmax(logits, dim=1)

predicted_label = int(probabilities.argmax(dim=1).item())
true_label = int(y.item())

print('File:', record['path'])
print('Nhãn thật:', CLASS_NAMES[true_label])
print('Logits:', logits.cpu().numpy()[0])
print('\nXác suất:')

for class_index, class_name in enumerate(CLASS_NAMES):
    probability = probabilities[0, class_index].item()
    print(f'{class_name:12s}: {probability*100:.10f}%')

print('\nDự đoán:', CLASS_NAMES[predicted_label])
print('Kết quả:', 'ĐÚNG' if predicted_label == true_label else 'SAI')